# Шаг 5. Свой «Инстаграм»: фильтры на веб-камере

Видео — это просто **последовательность кадров**, а каждый кадр — такая же картинка-массив, как фото.
Значит, любой фильтр для фото работает и на видео: достаточно применять его к каждому кадру в цикле.

> **Это версия для Google Colab.** Кадры снимает браузер и передаёт на сервер Colab, поэтому видео идёт медленнее и рывками.
> Когда браузер спросит про камеру, нажмите «Разрешить». На своём компьютере откройте `05_webcam_filters.ipynb`.

**Управление:** кнопки под видео переключают фильтр, **Сохранить** — сохранить кадр, **Стоп** — выход.

**Структура ноутбука (алгоритм):**
1. Импортируем OpenCV, помощник `label` и камеру для Colab.
2. Описываем фильтры как функции «кадр → кадр»: серый, карандашный рисунок, мультфильм и др.
3. Собираем словарь `FILTERS`: название → функция.
4. В цикле: получаем кадр → применяем текущий фильтр → подписываем → показываем → обрабатываем кнопку.
5. По кнопке «Стоп» цикл заканчивается, а камера выключается.

In [ ]:
# Настройка для Google Colab. На своём компьютере эта ячейка ничего не делает.
# В Colab нужна папка machine_vision на Google Диске (в ней session01–session05 и dataset).
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive:
    drive.mount("/content/drive")
    COLAB_ROOT = "/content/drive/MyDrive/machine_vision"
    os.chdir(f"{COLAB_ROOT}/session02")  # здесь лежит common.py и сохраняются результаты
    sys.path.insert(0, os.getcwd())
    sys.path.append(os.path.abspath("../session01"))  # colab_camera.py

## 1. Импорт

Из `common` нужен только `label` — он рисует подпись с названием фильтра.
Из `colab_camera` (файл лежит в папке `session01`) берём камеру через браузер: `frames` выдаёт кадры, `show` их показывает.

In [ ]:
import cv2

from colab_camera import frames, show
from common import label

## 2. Фильтры-функции

Каждый фильтр — функция, которая принимает кадр `f` и возвращает обработанный кадр.

**`gray`** — перевод в оттенки серого.

**`sketch` — карандашный рисунок** (приём «осветление основы», *color dodge*):
1. переводим кадр в серый `g`;
2. делаем негатив `255 - g` и сильно размываем его (окно 21 × 21);
3. делим серое на «негатив размытого негатива»: `g / (255 - blurred) · 256`.

На ровных участках числитель и знаменатель почти равны → результат ≈ 255 (белая бумага).
На границах, где яркость резко меняется, размытая версия отличается от исходной → остаются тёмные «штрихи».

**`cartoon` — мультфильм**:
1. `bilateralFilter` сильно сглаживает цвета, но сохраняет границы — получаются ровные «заливки»;
2. `medianBlur` убирает мелкий шум с серой версии, а `adaptiveThreshold` превращает её в чёрно-белую маску:
   для каждого пикселя порог считается **по его окрестности 9 × 9** (среднее минус 2), поэтому линии находятся
   и в тёмных, и в светлых местах кадра. Тёмные линии контуров — 0, остальное — 255;
3. `bitwise_and(colors, colors, mask=edges)` оставляет цвет только там, где маска белая, — контуры становятся чёрными.

In [ ]:
def gray(f):
    return cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)


def sketch(f):
    """Карандашный рисунок: серое делим на размытый негатив."""
    g = gray(f)
    blurred = cv2.GaussianBlur(255 - g, (21, 21), 0)
    return cv2.divide(g, 255 - blurred, scale=256)


def cartoon(f):
    """Мультфильм: гладкие цвета + чёрные контуры."""
    colors = cv2.bilateralFilter(f, 9, 150, 150)
    edges = cv2.adaptiveThreshold(cv2.medianBlur(gray(f), 7), 255,
                                  cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 2)
    return cv2.bitwise_and(colors, colors, mask=edges)

## 3. Словарь фильтров

Ключ — название фильтра, оно же надпись на кнопке. Значение — функция.

Простые фильтры записаны как `lambda` — короткие безымянные функции в одну строку:
- `mirror` — `cv2.flip(f, 1)` отражает кадр по горизонтали (как зеркало);
- `smooth` — сильное размытие Гаусса;
- `edges` — границы Canny.

Благодаря словарю добавить новый фильтр — это одна строка, основной цикл менять не нужно: кнопка появится сама.

In [ ]:
FILTERS = {
    "original": lambda f: f,
    "grayscale": gray,
    "negative": lambda f: 255 - f,
    "smooth": lambda f: cv2.GaussianBlur(f, (21, 21), 0),
    "edges": lambda f: cv2.Canny(gray(f), 60, 120),
    "mirror": lambda f: cv2.flip(f, 1),
    "sketch": sketch,
    "cartoon": cartoon,
}

## 4. Запускаем цикл

`frames(buttons=...)` включает камеру в браузере и добавляет кнопки. На каждом проходе цикла:
1. Получаем кадр `frame` и название нажатой кнопки `pressed` (если ничего не нажато — `None`).
2. Кнопка из словаря — сменить фильтр; `Сохранить` — сохранить кадр в файл
   `filter_01.jpg`, `filter_02.jpg`, … (`{shot:02d}` — номер с ведущим нулём).
3. `FILTERS[name](frame)` применяем текущий фильтр, `label` подписывает результат, `show` показывает кадр.

Кнопка «Стоп» заканчивает цикл, и камера выключается сама.

In [ ]:
name = "original"
shot = 0
for frame, pressed in frames(buttons=[*FILTERS, "Сохранить"]):
    if pressed in FILTERS:
        name = pressed

    picture = label(FILTERS[name](frame), name)
    show(picture)

    if pressed == "Сохранить":
        shot += 1
        cv2.imwrite(f"filter_{shot:02d}.jpg", picture)

## Попробуйте сами

1. Добавьте девятый фильтр — «сепия» или «ночное видение» (только зелёный канал: `f * [0, 1, 0]` с приведением к `uint8`).
2. Сделайте фильтр, который совмещает `mirror` и `cartoon`.
3. Выведите на кадр текущий FPS (число кадров в секунду). Подсказка: `time.perf_counter()`.
4. Почему `sketch` и `edges` возвращают серую картинку, а `label` всё равно рисует подпись? Посмотрите `to_bgr` в `common.py`.